# Proteins Mosaic Q — Try It Yourself

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UPO-Sevilla-Fco-Javier-Lobo-Cabrera/clustering_trait_proteins/blob/main/proteins_mosaic_q_demo.ipynb)

This notebook lets you compute the **Mosaic Q** parameter on any protein structure from the [Protein Data Bank](https://www.rcsb.org/), and visualize the mosaic in 3D, with no local setup. Just run the cells in order.

Companion article: *We Analyzed 160,000 Protein Structures with Python and Found a Surprising Pattern* (Towards Data Science).

Project: [proteins-mosaic-q.org](https://proteins-mosaic-q.org)

## 1. Install the libraries

Two packages: `protein-mosaic-q` for the Q calculation, and `py3Dmol` to render the protein in 3D inside the notebook.

In [ ]:
!pip install protein-mosaic-q py3Dmol --quiet

## 2. Download a famous protein

Let's start with **hemoglobin** (PDB code `4HHB`), the oxygen-carrying protein in your blood.

In [ ]:
import urllib.request

pdb_id = "4HHB"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
urllib.request.urlretrieve(url, f"{pdb_id}.pdb")
print(f"Downloaded {pdb_id}.pdb")

## 3. Visualize the mosaic in 3D

Each amino acid is rendered as space-filling (van der Waals) spheres and colored according to its chemical family, using exactly the same color scheme as the Jmol script on the [participation page](https://proteins-mosaic-q.org/participate/). The mosaic-like clustering of same-colored residues is what we are looking for.

**Color scheme** (matching the project's Jmol commands):

- ⚪ **Hydrophobic** (white): Ala, Val, Ile, Leu, Met, Phe, Tyr, Trp
- 🟢 **Polar** (green): Ser, Thr, Asn, Gln
- 🟠 **Acidic** (orange): Asp, Glu
- 🔵 **Basic** (blue): Arg, His, Lys
- 🩵 **Special** (cyan): Cys, Sec, Gly, Pro

Non-protein atoms (water, ligands, ions) are hidden, equivalent to Jmol's `restrict protein`.

*Drag to rotate, scroll to zoom.*

In [ ]:
import py3Dmol

# Color scheme matches the project's Jmol commands exactly:
#   ala, val, ile, leu, met, phe, tyr, trp  -> white
#   ser, thr, asn, gln                       -> green
#   asp, glu                                 -> orange
#   arg, his, lys                            -> blue
#   cys, sec, gly, pro                       -> cyan
CHEMICAL_FAMILIES = [
    (["ALA", "VAL", "ILE", "LEU", "MET", "PHE", "TYR", "TRP"], "white"),
    (["SER", "THR", "ASN", "GLN"],                              "green"),
    (["ASP", "GLU"],                                              "orange"),
    (["ARG", "HIS", "LYS"],                                       "blue"),
    (["CYS", "SEC", "GLY", "PRO"],                                "cyan"),
]

def show_mosaic(pdb_file, width=700, height=500):
    """Render a protein in space-filling style with each residue colored by its
    chemical family, equivalent to the project's Jmol script with `restrict protein`."""
    with open(pdb_file) as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_data, "pdb")

    # Start with no style for any atom,so non-protein atoms (waters, ligands,
    # ions) stay invisible.
    view.setStyle({}, {})

    # Apply space-filling spheres only to protein residues, colored by family.
    for residues, color in CHEMICAL_FAMILIES:
        view.setStyle({"resn": residues}, {"sphere": {"color": color}})

    view.zoomTo()
    view.setBackgroundColor("black")  # matches Jmol's default background
    return view.show()

show_mosaic("4HHB.pdb")

Look closely. Residues of the same color tend to form small clusters rather than being randomly scattered. That visual texture is what we call the **Mosaic Q**.

## 4. Quantify it: compute Q and Q_alt

- **Q** considers the four main chemical families: hydrophobic, polar, acidic, basic.
- **Q_alt** also includes the special amino acids: cysteine, glycine, proline, selenocysteine.

In [ ]:
from mosaicq import calculate_q, calculate_q_alt

q     = calculate_q("4HHB.pdb")
q_alt = calculate_q_alt("4HHB.pdb")

print(f"Hemoglobin (4HHB):")
print(f"  Q     = {q:.4f}")
print(f"  Q_alt = {q_alt:.4f}")

## 5. Visualize & Quantify the Mosaic Q on more famous proteins

Let's run the same calculation on a small set of well-known proteins and compare.

In [ ]:
proteins = {
    "4INS": "Insulin (blood sugar regulation)",
    "4UN3": "CRISPR-Cas9 protein",
    "1EMA": "Green Fluorescent Protein",
    "2LYZ": "Lysozyme (antibacterial enzyme)",
    "8RUC": "RuBisCO (CO2 fixation in photosynthesis)",
}

from IPython.display import display, Markdown

results = []
for pdb_id, description in proteins.items():
    urllib.request.urlretrieve(
        f"https://files.rcsb.org/download/{pdb_id}.pdb",
        f"{pdb_id}.pdb"
    )
    q     = calculate_q(f"{pdb_id}.pdb")
    q_alt = calculate_q_alt(f"{pdb_id}.pdb")
    results.append((pdb_id, description, q, q_alt))

    display(Markdown(f"### {pdb_id}. {description}"))
    display(Markdown(f"**Q = {q:.4f}** · **Q_alt = {q_alt:.4f}**"))
    show_mosaic(f"{pdb_id}.pdb", width=550, height=400)

## What now?

- **Read the full analysis**: companion article on Towards Data Science.
- **See the preprint and updated manuscript**: [bioRxiv](https://www.biorxiv.org/content/10.1101/500025v1.full) · [GitHub manuscript (latest version)](https://github.com/UPO-Sevilla-Fco-Javier-Lobo-Cabrera/clustering_trait_proteins/blob/main/Manuscript_and_Supplementary_Materials_v4.pdf).
- **Contribute a visualization**: render a protein in Jmol with the official color commands and submit it to the [visual repository](https://proteins-mosaic-q.org/repository/).
- **Use Galaxy** (no install needed at all): [Galaxy Europe tool](https://usegalaxy.eu/?tool_id=toolshed.g2.bx.psu.edu%2Frepos%2Fgalaxyp%2Fprotein_mosaic_q%2Fprotein_mosaic_q%2F0.3.3%2Bgalaxy0&version=latest).

### About the project

The Proteins Mosaic Q Project is an open citizen science initiative based in Seville, Spain. It has been reviewed and accepted onto the main international citizen science platforms:

- [SciStarter](https://scistarter.org/proteins-mosaic-q-project); global citizen science platform aggregating projects from NASA and other institutions.
- [EU-Citizen.Science](https://citizenscience.eu/project/686): the European citizen science platform, funded by the European Union through Horizon 2020 and Horizon Europe.
- [Observatorio de Ciencia Ciudadana](https://ciencia-ciudadana.es/project/392); the Spanish citizen science observatory.

It is also indexed by [bio.tools](https://bio.tools/protein-mosaic-q), [FAIRsharing](https://fairsharing.org/8206), and [Proteopedia](https://proteopedia.org/wiki/index.php/Mosaic_Q). The Python library is open source under the MIT license.

Project website: [proteins-mosaic-q.org](https://proteins-mosaic-q.org)